# Demonstração da múltiplicação da chave pela matriz.
Link: https://colab.research.google.com/drive/1Au51twbM_XO26zJ2HUaBwKoFbwDhTXSZ?usp=sharing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================================
# Parâmetros Iniciais (Hardcoded)
# =========================================================================
N = 128
L = 64

# Valores Hardcoded
key_hex  = "2CD"
seed_hex = "2A76"

# =========================================================================
# Conversão de Hexadecimal para Binário (Atenção ao Endianness)
# =========================================================================
# O bit 0 das listas será o LSB (Least Significant Bit), igual ao Verilog.
key_int = int(key_hex, 16)
seed_int = int(seed_hex, 16)

# Extração dos 128 bits da chave
key_bits = np.array([(key_int >> i) & 1 for i in range(N)])

# O tamanho da semente que entra no shift é N + L - 1 (191 bits).
# O crop é feito implicitamente ao limitar o range do loop.
TOTAL_SEED_BITS = N + L - 1
seed_bits = np.array([(seed_int >> i) & 1 for i in range(TOTAL_SEED_BITS)])

# =========================================================================
# Geração da Matriz Dinâmica
# =========================================================================
# O Verilog usa a janela HARDCODED_SEED[seed_offset +: (W+P-1)].
# Juntando offset e instâncias, a posição da matriz recai exatamente sobre r + c.
matrix = np.zeros((L, N), dtype=int)
for r in range(L):
    for c in range(N):
        matrix[r, c] = seed_bits[r + c]

# =========================================================================
# Operação GF(2): AND + Árvore de XOR
# =========================================================================
result_bits = np.zeros(L, dtype=int)
for r in range(L):
    # AND bit a bit com a chave, e depois soma módulo 2 (XOR)
    result_bits[r] = np.sum(key_bits & matrix[r]) % 2

# =========================================================================
# Reconstrução do Resultado Final (Binário para Hexadecimal)
# =========================================================================
result_int = 0
for i in range(L):
    result_int |= (int(result_bits[i]) << i)

# Formatação padronizada para bater com a exibição do testbench
hex_length = L // 4
result_hex_str = f"{result_int:0{hex_length}X}"

# =========================================================================
# Print no Terminal de Saída
# =========================================================================
print("="*60)
print(" RESULTADOS DA VALIDAÇÃO ALTO NÍVEL (GF2 - TOEPLITZ)")
print("="*60)
print(f"Key  (N={N})    : {key_hex}")
print(f"Seed (N+L-1={TOTAL_SEED_BITS}): {seed_hex}")
print("-" * 60)
print(f"RESULTADO FINAL : {result_hex_str}")
print("="*60)

# =========================================================================
# Visualização Gráfica
# =========================================================================
# Configuração da figura para desenhar a Key, a Matrix e o Resultado
fig, axes = plt.subplots(1, 3, figsize=(12, 6), gridspec_kw={'width_ratios': [1, 4, 1]})

# Array da Key (Desenhado em coluna para mapear visualmente)
ax_key = axes[0]
ax_key.imshow(key_bits.reshape(-1, 1), cmap='Blues', aspect='auto')
ax_key.set_title(f"Key Array\n({N}x1)")
ax_key.set_ylabel("Index (Top = LSB)")
ax_key.set_xticks([])

# Matriz Gerada da Semente
ax_mat = axes[1]
ax_mat.imshow(matrix, cmap='Greens', aspect='auto')
ax_mat.set_title(f"Toeplitz Matrix\n({L}x{N})")
ax_mat.set_xlabel("Key Index (Left = LSB)")
ax_mat.set_ylabel("Result Row Index (Top = LSB)")

# Vetor de Resultado
ax_res = axes[2]
ax_res.imshow(result_bits.reshape(-1, 1), cmap='Reds', aspect='auto')
ax_res.set_title(f"Result\n({L}x1)")
ax_res.set_xticks([])
ax_res.set_yticks([])

plt.tight_layout()
plt.show()